# Thai License Plate OCR — Colab walkthrough

Two-stage Thai license-plate OCR: YOLOv8 plate detector + YOLOv8 per-character detector, with spatial post-processing for both single- and two-line Thai plates.

Data is pulled from two public CC BY 4.0 Roboflow Universe datasets.

**You will need:** a free Roboflow account and API key from [app.roboflow.com/settings/api](https://app.roboflow.com/settings/api). A T4 GPU (free Colab) is strongly recommended — Runtime → Change runtime type → T4 GPU.

**End-to-end time on a T4:** dataset download + both training stages + evaluation ≈ 25–35 minutes.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'Switch runtime to GPU: Runtime -> Change runtime type -> T4 GPU'

In [ ]:
!git clone --depth 1 https://github.com/simonyos/thai-plate-ocr.git
%cd thai-plate-ocr
!pip install -q -e . --no-deps
!pip install -q numpy pandas pillow opencv-python-headless ultralytics torch torchvision roboflow matplotlib mlflow typer rich fastapi 'uvicorn[standard]' python-multipart pydantic tqdm tabulate pyyaml

In [ ]:
import os, getpass
os.environ['ROBOFLOW_API_KEY'] = getpass.getpass('Paste your Roboflow API key: ')

## 1. Download both datasets

In [ ]:
!plate download

## 2. Train stage 1 (plate detection)

In [ ]:
!plate train-detector

## 3. Train stage 2 (character detection)

In [ ]:
!plate train-recognizer

## 4. Evaluate + see summary

In [ ]:
!plate evaluate
from IPython.display import Image, Markdown, display
display(Markdown(open('reports/summary.md').read()))
display(Image('reports/figures/map_by_stage.png'))

## 5. End-to-end prediction on a sample image

In [ ]:
from pathlib import Path
import random
from PIL import Image, ImageDraw
from thai_plate_ocr.pipeline import PlatePipeline

det = Path('artifacts/detector/train/weights/best.pt')
rec = Path('artifacts/recognizer/train/weights/best.pt')
pipe = PlatePipeline(det, rec)

candidates = list(Path('data/detector').rglob('*.jpg'))
img_path = random.choice(candidates)
img = Image.open(img_path).convert('RGB')
preds = pipe.predict(img)

draw = ImageDraw.Draw(img)
for p in preds:
    draw.rectangle(p.plate_bbox_xyxy, outline='red', width=3)
    draw.text((p.plate_bbox_xyxy[0], max(0, p.plate_bbox_xyxy[1] - 18)), p.text, fill='red')
img

## 6. Package results for download

Zips the trained weights, per-stage summaries, reports and a couple of diagnostic figures into a single small archive, then triggers a browser download. After the download finishes, on your laptop:

```bash
cd ~/Development/thai-plate-ocr
unzip -o ~/Downloads/thai_plate_ocr_artifacts.zip
make serve    # demo the FastAPI endpoint locally
```

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

zip_path = 'thai_plate_ocr_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root in ('artifacts', 'reports'):
        for p in Path(root).rglob('*'):
            if not p.is_file():
                continue
            if p.suffix in ('.pt', '.json', '.png', '.md') or 'best' in p.name:
                zf.write(p, p.as_posix())
size_mb = Path(zip_path).stat().st_size / 1e6
print(f'Packaged {zip_path}: {size_mb:.1f} MB')
files.download(zip_path)

## 7. (optional) Commit results back to the repo

If you'd rather commit the small artifacts (reports + summaries, *not* the large weights) straight to GitHub from Colab, set a PAT with `contents:write` scope in `GH_TOKEN` and run the cell below.

```python
# import os, getpass, subprocess
# os.environ['GH_TOKEN'] = getpass.getpass('GitHub PAT (contents:write): ')
# subprocess.run(['git', 'config', 'user.email', 'simon.y@realfactory.co.th'], check=True)
# subprocess.run(['git', 'config', 'user.name',  'Simon Yosboon'], check=True)
# subprocess.run(['git', 'add', 'reports/'], check=True)
# subprocess.run(['git', 'commit', '-m', 'first end-to-end run on Colab (T4)'], check=True)
# subprocess.run(['git', 'push', f'https://{os.environ["GH_TOKEN"]}@github.com/simonyos/thai-plate-ocr.git', 'HEAD:main'], check=True)
```